In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("LLM Reliability Analysis with Metal GPU")
print("=" * 70)

import tensorflow as tf

print(f"\nTensorFlow: {tf.__version__}")
print(f"Device: Apple M2")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Status: ACTIVE")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"   - Memory growth: ENABLED")

    with tf.device('/GPU:0'):
        test_tensor = tf.random.normal([100, 100])
        test_result = tf.reduce_sum(test_tensor)
        print(f"   - GPU test: PASSED")
else:
    print("GPU not detected - running on CPU")

LLM Reliability Analysis with Metal GPU

TensorFlow: 2.13.0
Device: Apple M2
GPU Status: ACTIVE
   - Memory growth: ENABLED
   - GPU test: PASSED


2026-03-23 21:35:25.633510: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-03-23 21:35:25.633528: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-03-23 21:35:25.633533: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-03-23 21:35:25.633874: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-23 21:35:25.634229: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [41]:
import wikipedia
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import requests
import torch
from ddgs import DDGS
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from functools import lru_cache

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        # Prevent TF from pre-allocating all memory to allow Ollama to run alongside it
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        print(f"TensorFlow is accelerated by: {physical_devices[0]}")
    except RuntimeError as e:
        print(f"GPU Configuration Error: {e}")
else:
    print("Metal GPU not detected. Ensure 'tensorflow-metal' is installed in your 3.11 kernel.")

TensorFlow is accelerated by: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [21]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
nli_analyzer = pipeline(
    "text-classification", 
    model="roberta-large-mnli", 
    framework="tf"
)
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:3b"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
@lru_cache(maxsize=128)
def get_automated_reference(prompt):
    """Returns (reference_text, is_verified) tuple."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(prompt, max_results=1, timeout=5))
            if results:
                snippet = f"{results[0]['title']}: {results[0]['body']}"
                return snippet, True
    except Exception:
        pass

    try:
        import socket
        socket.setdefaulttimeout(5)
        text = wikipedia.summary(prompt, sentences=2)
        return text, True
    except Exception:
        pass

    return "No verifiable reference found.", False

In [34]:
# Cell 9 — GPU-accelerated embedding & hallucination penalty

@tf.function
def tf_cosine_similarity(a, b):
    """Cosine similarity on GPU."""
    a = tf.cast(a, tf.float32)
    b = tf.cast(b, tf.float32)
    dot = tf.reduce_sum(a * b)
    norm = tf.norm(a) * tf.norm(b)
    return tf.where(norm < 1e-8, tf.zeros_like(dot), dot / norm)

@lru_cache(maxsize=256)
def get_tf_embedding(text: str):
    """Encode text to a TF tensor (cached to avoid re-encoding same strings)."""
    vec = embedding_model.encode(text, convert_to_numpy=True)
    return tf.constant(vec, dtype=tf.float32)

def check_hallucination_penalty(response: str, reference: str) -> float:
    """
    Returns a penalty in [0, 1].
    High penalty = response likely hallucinated relative to reference.
    Uses NLI: if the model output contradicts the reference, penalise heavily.
    """
    if not reference or "No verifiable reference" in reference:
        return 0.0  # can't penalise without a ground truth

    try:
        # Truncate to avoid NLI token limits
        premise    = reference[:400]
        hypothesis = response[:200]
        result = nli_analyzer(f"{premise} [SEP] {hypothesis}", truncation=True)[0]
        label  = result["label"].upper()
        score  = result["score"]

        if label == "CONTRADICTION":
            return score * 0.9          # strong penalty
        elif label == "NEUTRAL":
            return score * 0.2          # mild penalty
        else:                           # ENTAILMENT
            return 0.0
    except Exception as e:
        print(f"NLI check failed: {e}")
        return 0.1                      # small default penalty on failure

In [25]:
def run_model(prompt, temperature=0.7):
    """Query local Ollama with temperature for varied responses."""
    start_time = time.time()
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 60,
            "temperature": temperature,      # <--- added
        }
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=10)
        if response.status_code == 200:
            output_text = response.json().get("response", "")
        else:
            output_text = f"[Ollama error: status {response.status_code}]"
    except Exception as e:
        output_text = f"[Ollama error: {e}]"
    return output_text, (time.time() - start_time)

In [14]:
# Alias for embedding function
get_embedding = get_tf_embedding

def semantic_similarity(model_output, reference):
    """Compute similarity using GPU ops."""
    try:
        emb1 = get_embedding(model_output)
        emb2 = get_embedding(reference)
        return float(tf_cosine_similarity(emb1, emb2))
    except Exception as e:
        print(f"Similarity calculation failed: {e}")
        return 0.5

def evidence_score(response):
    """Compare model response with a Wikipedia summary, with timeout."""
    try:
        query = response.split(".")[0][:100]
        # Use the same cached reference approach; avoid repeated network calls
        wiki_text = get_automated_reference(query)  # reuse caching
        if "No verifiable reference" in wiki_text:
            return 0.5
        emb1 = get_embedding(response)
        emb2 = get_embedding(wiki_text)
        return float(tf_cosine_similarity(emb1, emb2))
    except Exception as e:
        print(f"Evidence retrieval failed: {e}")
        return 0.5

def self_consistency(outputs):
    """Compute average pairwise similarity among multiple outputs using GPU ops."""
    if len(outputs) < 2:
        return 1.0
    try:
        embeddings = [get_embedding(o) for o in outputs]
        sims = []
        for i in range(len(embeddings)):
            for j in range(i + 1, len(embeddings)):
                sim = tf_cosine_similarity(embeddings[i], embeddings[j])
                sims.append(float(sim))
        return np.mean(sims) if sims else 1.0
    except Exception as e:
        print(f"Consistency calculation failed: {e}")
        return 0.7

In [36]:
def evaluate_prompt(prompt, reference=None):
    print(f"\nProcessing: {prompt[:80]}...")
    
    if reference is not None:
    ref, ref_verified = reference, True
else:
    ref, ref_verified = get_automated_reference(prompt)

# Skip hallucination penalty when reference is unverified
h_penalty = check_hallucination_penalty(main_output, ref) if ref_verified else 0.0
    outputs, latencies = [], []
    print("   Responses:")
    for i in range(3):
        resp, lat = run_model(prompt, temperature=0.8)   # use a bit of randomness
        outputs.append(resp)
        latencies.append(lat)
        print(f"      {i+1}: {resp[:100]}{'...' if len(resp) > 100 else ''}")
    
    main_output = outputs[0]
    
    sim_score = semantic_similarity(main_output, ref)
    evidence = evidence_score(main_output)
    consistency = self_consistency(outputs)
    h_penalty = check_hallucination_penalty(main_output, ref)
    
    base_reliability = (0.35 * sim_score + 0.35 * evidence + 0.3 * consistency)
    final_score = base_reliability - (0.3 * h_penalty)
    final_score = max(0, min(1, final_score))
    
    print(f"   Sim={sim_score:.3f}, Evid={evidence:.3f}, Cons={consistency:.3f}, Hall={h_penalty:.3f} -> Final={final_score:.3f}")
    
    return {
        "response": main_output,
        "reference": ref,
        "similarity": sim_score,
        "evidence": evidence,
        "consistency": consistency,
        "h_penalty": h_penalty,
        "latency": np.mean(latencies),
        "final": final_score
    }

IndentationError: expected an indented block after 'if' statement on line 4 (284856370.py, line 5)

In [10]:
dataset = [

# -------- Geography --------
{"prompt":"What is the capital of Japan?","reference":"Tokyo"},
{"prompt":"What is the capital of France?","reference":"Paris"},
{"prompt":"What is the capital of Canada?","reference":"Ottawa"},
{"prompt":"What is the capital of Australia?","reference":"Canberra"},
{"prompt":"What is the capital of Germany?","reference":"Berlin"},
{"prompt":"Which is the largest country by area?","reference":"Russia"},
{"prompt":"Which ocean is the largest?","reference":"Pacific Ocean"},
{"prompt":"Which desert is the largest hot desert?","reference":"Sahara Desert"},
{"prompt":"Which river is the longest in the world?","reference":"Nile River"},
{"prompt":"Mount Everest lies in which mountain range?","reference":"Himalayas"},
{"prompt":"Which country has the largest population?","reference":"China"},
{"prompt": "What is the current estimated population of Gwalior as of 2024?"},
{"prompt": "What is the newest state or administrative territory created in India?"},
{"prompt":"Which continent is the Sahara Desert located in?","reference":"Africa"},

# -------- Mathematics --------
{"prompt":"What is 12 multiplied by 8?","reference":"96"},
{"prompt":"What is the square root of 144?","reference":"12"},
{"prompt":"Solve for x: 2x + 6 = 14","reference":"x = 4"},
{"prompt":"What is 15 percent of 200?","reference":"30"},
{"prompt":"What is the value of pi approximately?","reference":"3.1416"},
{"prompt":"What is the cube of 5?","reference":"125"},
{"prompt":"What is 7 squared?","reference":"49"},
{"prompt": "What is the 500th prime number?"},
{"prompt": "What is the current value of the 100th digit of Pi?"},
{"prompt":"What is the derivative of x^2?","reference":"2x"},
{"prompt":"What is the integral of 1/x?","reference":"ln(x)"},
{"prompt":"What is 100 divided by 4?","reference":"25"},

# -------- Science --------
{"prompt":"What gas do humans breathe out?","reference":"Carbon dioxide"},
{"prompt":"What is the chemical symbol for water?","reference":"H2O"},
{"prompt":"What planet is known as the Red Planet?","reference":"Mars"},
{"prompt": "What are the latest findings from the James Webb Space Telescope regarding the atmosphere of exoplanet WASP-39b?"},
{"prompt": "Who were the winners of the 2024 Nobel Prize in Physics and what was their specific contribution?"},
{"prompt":"What is the speed of light approximately?","reference":"299792458 meters per second"},
{"prompt":"What force pulls objects toward Earth?","reference":"Gravity"},
{"prompt":"What part of the cell contains genetic material?","reference":"Nucleus"},
{"prompt":"What gas do plants absorb from the atmosphere?","reference":"Carbon dioxide"},
{"prompt":"What is the boiling point of water at sea level?","reference":"100 degrees Celsius"},
{"prompt":"What is the freezing point of water?","reference":"0 degrees Celsius"},
{"prompt":"Which organ pumps blood in the human body?","reference":"Heart"},
{"prompt":"Which planet is the largest in the solar system?","reference":"Jupiter"},
{"prompt":"Which particle carries a negative charge?","reference":"Electron"},

# -------- History --------
{"prompt":"Who wrote Romeo and Juliet?","reference":"William Shakespeare"},
{"prompt":"Who was the first President of the United States?","reference":"George Washington"},
{"prompt":"In which year did World War II end?","reference":"1945"},
{"prompt":"Who discovered penicillin?","reference":"Alexander Fleming"},
{"prompt":"Who painted the Mona Lisa?","reference":"Leonardo da Vinci"},
{"prompt":"Which ancient civilization built the pyramids?","reference":"Ancient Egyptians"},
{"prompt":"Who was known as the Maid of Orleans?","reference":"Joan of Arc"},
{"prompt":"Who invented the telephone?","reference":"Alexander Graham Bell"},
{"prompt":"Who was the first man to walk on the Moon?","reference":"Neil Armstrong"},
{"prompt": "What were the key archeological discoveries made at the Rakhigarhi site during the 2023-2024 excavation season?"},
{"prompt": "What is the current status of the restoration project for the Notre-Dame de Paris as of early 2024?"},
{"prompt":"Which empire was ruled by Julius Caesar?","reference":"Roman Empire"},

# -------- Technology --------
{"prompt":"Who founded Microsoft?","reference":"Bill Gates"},
{"prompt":"What does CPU stand for?","reference":"Central Processing Unit"},
{"prompt":"What does HTTP stand for?","reference":"Hypertext Transfer Protocol"},
{"prompt":"What programming language is commonly used for machine learning?","reference":"Python"},
{"prompt":"Who founded Tesla?","reference":"Elon Musk"},
{"prompt": "What are the latest performance benchmarks for the Apple M3 Max chip compared to the M2 Ultra?"},
{"prompt": "What is the current version and most recent feature update of the Mojo programming language?"},
{"prompt":"What does RAM stand for?","reference":"Random Access Memory"},
{"prompt":"Which company developed the iPhone?","reference":"Apple"},
{"prompt":"What does GPU stand for?","reference":"Graphics Processing Unit"},
{"prompt":"Which language is used for styling web pages?","reference":"CSS"},
{"prompt":"Which protocol secures web communication?","reference":"HTTPS"},

# -------- Reasoning --------
{"prompt":"If a car travels 60 km per hour for 3 hours how far does it travel?","reference":"180 km"},
{"prompt":"If you have 10 apples and give away 4 how many remain?","reference":"6"},
{"prompt": "Given the current weather in Indore today, would it be advisable to wear a heavy wool coat?"},
{"prompt": "If a flight left New York for London two hours ago, what is its current approximate position?"},
{"prompt":"If a triangle has angles 90, 45 and 45 what type is it?","reference":"Right triangle"},
{"prompt":"If 5 machines make 5 parts in 5 minutes how long for 100 machines to make 100 parts?","reference":"5 minutes"},
{"prompt":"If today is Monday what day will it be in 3 days?","reference":"Thursday"},
{"prompt":"If a rectangle has length 10 and width 5 what is the area?","reference":"50"},
{"prompt":"If a train travels 50 km in 1 hour how far in 4 hours?","reference":"200 km"},
{"prompt":"If the perimeter of a square is 20 what is its side length?","reference":"5"},
{"prompt":"If you double 8 what do you get?","reference":"16"},
{"prompt":"If a dozen eggs cost 12 dollars what is the price per egg?","reference":"1 dollar"},

# -------- Conceptual --------
{"prompt":"What is photosynthesis?","reference":"Process by which plants convert sunlight into chemical energy"},
{"prompt":"What is gravity?","reference":"Force that attracts objects with mass toward each other"},
{"prompt":"What is machine learning?","reference":"Field of AI where computers learn patterns from data"},
{"prompt":"What is artificial intelligence?","reference":"Simulation of human intelligence in machines"},
{"prompt":"What is a black hole?","reference":"Region of spacetime with gravity so strong that nothing escapes"},
{"prompt":"What is DNA?","reference":"Molecule carrying genetic information"},
{"prompt":"What is an algorithm?","reference":"Step by step procedure to solve a problem"},
{"prompt":"What is cloud computing?","reference":"Delivery of computing services over the internet"},
{"prompt":"What is the internet?","reference":"Global network of interconnected computers"},
{"prompt":"What is blockchain?","reference":"Distributed ledger technology for secure transactions"},
{"prompt": "What is the current scientific consensus on 'Room-Temperature Superconductivity' following the LK-99 papers?"},
{"prompt": "What are the most recent ethical guidelines published by the EU regarding Generative AI development in 2024?"},

# -------- Hallucination traps --------
{"prompt":"Who was the president of the United States in 1785?","reference":"There was no US president in 1785"},
{"prompt":"What is the capital of the fictional country Wakanda?","reference":"Wakanda is fictional"},
{"prompt":"Who invented the electric spoon in 1889?","reference":"No known inventor"},
{"prompt":"What is the capital city of Atlantis?","reference":"Atlantis is mythical"},
{"prompt":"Who discovered the planet Vulcan in 1915?","reference":"Planet Vulcan does not exist"},
{"prompt":"Which scientist discovered gravity in 1805?","reference":"Gravity was not discovered in 1805"},
{"prompt":"What year was the invisible city of Eloria founded?","reference":"Eloria is fictional"},
{"prompt":"Who wrote the book Shadows of Jupiter in 1750?","reference":"No such book exists"},
{"prompt":"Which country owns the Moon?","reference":"No country owns the Moon"},
{"prompt":"Which company invented teleportation technology?","reference":"Teleportation technology does not exist"},

# -------- Unanswerable --------
{"prompt":"What is the exact number of stars in the universe?","reference":"Unknown"},
{"prompt":"How many grains of sand exist on Earth?","reference":"Unknown"},
{"prompt":"What will be the population of Earth in 3000?","reference":"Unknown"},
{"prompt":"Which person will live the longest in the future?","reference":"Unknown"},
{"prompt":"When will humans colonize another galaxy?","reference":"Unknown"},
{"prompt":"What is the exact number of trees on Earth right now?","reference":"Unknown"},
{"prompt":"Which language will dominate the world in 500 years?","reference":"Unknown"},
{"prompt":"Which company will be the richest in 2100?","reference":"Unknown"},
{"prompt":"Which city will be the largest in 2200?","reference":"Unknown"},
{"prompt":"What technology will replace the internet?","reference":"Unknown"},
{"prompt": "Who is the current CEO of the startup 'Aetheria Neural Systems' founded in 2024?"},
{"prompt": "Which major city hosted the Intergalactic Peace Summit in January 2024?"}

]

In [44]:
records = []


for i, item in enumerate(dataset):
    p, ref = item["prompt"], item.get("reference")
    print(f"\n{'='*70}")
    print(f"[{i+1}/{len(new_dataset)}] Prompt: {p}")
    print(f"{'='*70}")

    result = evaluate_prompt(p, ref)

    # Print side-by-side comparison
    print(f"\n  REFERENCE ANSWER:")
    print(f"  {result['reference'][:300]}{'...' if len(result['reference']) > 300 else ''}")
    print(f"\n  MODEL ANSWER:")
    print(f"  {result['response'][:300]}{'...' if len(result['response']) > 300 else ''}")
    print(f"\n  SCORES → Sim={result['similarity']:.3f} | Evid={result['evidence']:.3f} | "
          f"Cons={result['consistency']:.3f} | Hall={result['h_penalty']:.3f} | "
          f"Final={result['final']:.3f}")

    records.append({
        "Prompt":         p,
        "Reference":      result["reference"],        # ← added
        "Model_Response": result["response"],
        "Similarity":     round(result["similarity"],  3),
        "Evidence":       round(result["evidence"],    3),  # ← added
        "Consistency":    round(result["consistency"], 3),
        "H_Penalty":      round(result["h_penalty"],   3),
        "Final_Score":    result["final"]
    })

df_results = pd.DataFrame(records)

print(f"\n{'='*70}")
print("SUMMARY TABLE")
print(f"{'='*70}")

# Display with reference column visible
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 10)
print(df_results[["Prompt", "Reference", "Model_Response", 
                   "Similarity", "Consistency", "H_Penalty", "Final_Score"]].to_string(index=False))


[1/3] Prompt: What is the capital of Japan?

Processing: What is the capital of Japan?...
   Responses:
      1: The capital of Japan is Tokyo.
      2: The capital of Japan is Tokyo.
      3: The capital of Japan is Tokyo.
   Sim=0.700, Evid=0.590, Cons=1.000, Hall=0.000 -> Final=0.752

  REFERENCE ANSWER:
  Tokyo

  MODEL ANSWER:
  The capital of Japan is Tokyo.

  SCORES → Sim=0.700 | Evid=0.590 | Cons=1.000 | Hall=0.000 | Final=0.752

[2/3] Prompt: What is the capital of France?

Processing: What is the capital of France?...
   Responses:
      1: The capital of France is Paris.
      2: The capital of France is Paris.
      3: The capital of France is Paris.
   Sim=0.725, Evid=0.431, Cons=1.000, Hall=0.000 -> Final=0.705

  REFERENCE ANSWER:
  Paris

  MODEL ANSWER:
  The capital of France is Paris.

  SCORES → Sim=0.725 | Evid=0.431 | Cons=1.000 | Hall=0.000 | Final=0.705

[3/3] Prompt: What is the capital of Canada?

Processing: What is the capital of Canada?...
   Responses:
 

In [45]:
df = pd.DataFrame(records)
print("\nFinal Reliability Rankings:")
print(df.sort_values(by="Final_Score", ascending=False).head(20))


Final Reliability Rankings:
                                                     Prompt  \
3                         What is the capital of Australia?   
0                             What is the capital of Japan?   
2                            What is the capital of Canada?   
4                           What is the capital of Germany?   
1                            What is the capital of France?   
36                     What is the freezing point of water?   
8                  Which river is the longest in the world?   
38         Which planet is the largest in the solar system?   
76                                  What is photosynthesis?   
28                  What planet is known as the Red Planet?   
68  If a triangle has angles 90, 45 and 45 what type is it?   
79                         What is artificial intelligence?   
26                          What gas do humans breathe out?   
31                What is the speed of light approximately?   
96                        

In [20]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:3b"   # exact name from ollama list

payload = {
    "model": MODEL_NAME,
    "prompt": "What is the capital of Japan?",
    "stream": False,
    "options": {"num_predict": 50}
}

try:
    response = requests.post(OLLAMA_URL, json=payload, timeout=60)
    if response.status_code == 200:
        result = response.json()
        print("Response:", result.get("response", ""))
    else:
        print(f"HTTP {response.status_code}: {response.text}")
except Exception as e:
    print(f"Error: {e}")

Response: The capital of Japan is Tokyo.
